# Fase 1 - Modelo predictivo

## Predicción del precio unitario de pizzas

En este notebook se exploran y preparan los datos, se desarrolla un modelo de Machine Learning para predecir `unit_price`, se evalúa su desempeño y se almacena el modelo entrenado.


### Objetivos

- Explorar los datos.
- Preparar las variables para el modelo.
- Evitar fuga de información.
- Entrenar un modelo predictivo.
- Evaluar el modelo mediante RMSE, MAE y R².
- Guardar el modelo entrenado.


In [68]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


## 1. Cargar el dataset

El notebook está dentro de `fase-1`, por eso se utiliza `../data/` para acceder al dataset original.

In [69]:
df = pd.read_csv("../data/pizza_sales.csv")

print("Dimensiones del dataset:", df.shape)
df.head()


Dimensiones del dataset: (48620, 12)


,pizza_id,order_id,pizza_name_id,quantity,order_date,order_time,unit_price,total_price,pizza_size,pizza_category,pizza_ingredients,pizza_name
0,1.0,1.0,hawaiian_m,1.0,1/1/2015,11:38:36,13.25,13.25,M,Classic,"Sliced Ham, Pineapple, Mozzarella Cheese",The Hawaiian Pizza
1,2.0,2.0,classic_dlx_m,1.0,1/1/2015,11:57:40,16.00,16.00,M,Classic,"Pepperoni, Mushrooms, Red Onions, Red Peppers,...",The Classic Deluxe Pizza
2,3.0,2.0,five_cheese_l,1.0,1/1/2015,11:57:40,18.50,18.50,L,Veggie,"Mozzarella Cheese, Provolone Cheese, Smoked Go...",The Five Cheese Pizza
3,4.0,2.0,ital_supr_l,1.0,1/1/2015,11:57:40,20.75,20.75,L,Supreme,"Calabrese Salami, Capocollo, Tomatoes, Red Oni...",The Italian Supreme Pizza
4,5.0,2.0,mexicana_m,1.0,1/1/2015,11:57:40,16.00,16.00,M,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza


## 2. Exploración inicial

In [70]:
print("Columnas:")
print(df.columns.tolist())

print("\nValores faltantes:")
print(df.isnull().sum())

print("\nInformación del dataset:")
df.info()


Columnas:
['pizza_id', 'order_id', 'pizza_name_id', 'quantity', 'order_date', 'order_time', 'unit_price', 'total_price', 'pizza_size', 'pizza_category', 'pizza_ingredients', 'pizza_name']

Valores faltantes:
pizza_id             0
order_id             0
pizza_name_id        0
quantity             0
order_date           0
order_time           0
unit_price           0
total_price          0
pizza_size           0
pizza_category       0
pizza_ingredients    0
pizza_name           0
dtype: int64

Información del dataset:
<class 'pandas.DataFrame'>
RangeIndex: 48620 entries, 0 to 48619
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   pizza_id           48620 non-null  float64
 1   order_id           48620 non-null  float64
 2   pizza_name_id      48620 non-null  str    
 3   quantity           48620 non-null  float64
 4   order_date         48620 non-null  str    
 5   order_time         48620 non-null  str

In [71]:
df.describe(include="all").T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
pizza_id,48620.0,NaN,NaN,NaN,24310.5,14035.529381,1.0,12155.75,24310.5,36465.25,48620.0
order_id,48620.0,NaN,NaN,NaN,10701.479761,6180.11977,1.0,5337.0,10682.5,16100.0,21350.0
pizza_name_id,48620,91,big_meat_s,1811,NaN,NaN,NaN,NaN,NaN,NaN,NaN
quantity,48620.0,NaN,NaN,NaN,1.019622,0.143077,1.0,1.0,1.0,1.0,4.0
order_date,48620,358,26-11-2015,261,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_time,48620,16382,12:32:00,26,NaN,NaN,NaN,NaN,NaN,NaN,NaN
unit_price,48620.0,NaN,NaN,NaN,16.494132,3.621789,9.75,12.75,16.5,20.25,35.95
total_price,48620.0,NaN,NaN,NaN,16.821474,4.437398,9.75,12.75,16.5,20.5,83.0
pizza_size,48620,5,L,18526,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pizza_category,48620,4,Classic,14579,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Preparación de los datos

Se crean algunas variables relacionadas con los ingredientes. Las variables de identificación, las variables de texto originales y las variables temporales se eliminan porque no serán utilizadas por este modelo.

`total_price` también se elimina porque está calculado a partir de `unit_price` y `quantity`, por lo que utilizarlo para predecir `unit_price` podría introducir fuga de información.

In [72]:
df["n_ingredientes"] = df["pizza_ingredients"].fillna("").str.split(",").apply(
    lambda ingredientes: len([x for x in ingredientes if x.strip()])
)

df["tiene_pollo"] = df["pizza_ingredients"].fillna("").str.contains(
    "chicken", case=False, na=False
)

df["tiene_carnes"] = df["pizza_ingredients"].fillna("").str.contains(
    "pepperoni|bacon|sausage|salami", case=False, na=False
)

df["tiene_vegetales"] = df["pizza_ingredients"].fillna("").str.contains(
    "pepper|onion|mushroom|spinach|tomato", case=False, na=False
)


In [73]:
columnas_eliminar = [
    "pizza_id",
    "order_id",
    "pizza_name_id",
    "pizza_ingredients",
    "total_price",
    "order_date",
    "order_time"
]

# pizza_name se conserva como variable categórica para que el modelo
# pueda aprender diferencias de precio entre tipos de pizza.
df = df.drop(columns=columnas_eliminar, errors="ignore")

print("Columnas utilizadas:")
print(df.columns.tolist())


Columnas utilizadas:
['quantity', 'unit_price', 'pizza_size', 'pizza_category', 'pizza_name', 'n_ingredientes', 'tiene_pollo', 'tiene_carnes', 'tiene_vegetales']


## 4. Separar variables predictoras y objetivo

La variable objetivo es `unit_price`. Las demás variables disponibles serán utilizadas como variables predictoras.

In [74]:
X = df.drop(columns=["unit_price", "pizza_name"])
y = df["unit_price"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Datos de entrenamiento:", X_train.shape)
print("Datos de prueba:", X_test.shape)


Datos de entrenamiento: (38896, 7)
Datos de prueba: (9724, 7)


## 5. Preprocesamiento y modelo

Se utiliza un `Pipeline` para que las transformaciones se aprendan únicamente a partir de los datos de entrenamiento. Las variables numéricas reciben imputación de valores faltantes y las categóricas se codifican mediante One-Hot Encoding.

Esto permite mantener separado el proceso de entrenamiento y evaluación y reducir el riesgo de fuga de información.

In [75]:
columnas_numericas = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
columnas_categoricas = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

preprocesamiento = ColumnTransformer(
    transformers=[
        (
            "numericas",
            Pipeline([
                ("imputacion", SimpleImputer(strategy="median"))
            ]),
            columnas_numericas
        ),
        (
            "categoricas",
            Pipeline([
                ("imputacion", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            columnas_categoricas
        )
    ]
)

modelo = Pipeline([
    ("preprocesamiento", preprocesamiento),
    ("regresor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])


C:\Users\USER\AppData\Local\Temp\ipykernel_18652\2267748018.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_categoricas = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


## 6. Entrenamiento

In [76]:
modelo.fit(X_train, y_train)
print("Modelo entrenado correctamente.")


Modelo entrenado correctamente.


## 7. Evaluación del modelo

In [77]:
predicciones = modelo.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, predicciones))
mae = mean_absolute_error(y_test, predicciones)
r2 = r2_score(y_test, predicciones)

metricas = pd.DataFrame({
    "Metrica": ["RMSE", "MAE", "R2"],
    "Valor": [rmse, mae, r2]
})

metricas


,Metrica,Valor
0,RMSE,0.763985
1,MAE,0.135845
2,R2,0.955790


### Interpretación

- **RMSE** mide el error promedio del modelo dando mayor peso a errores grandes.
- **MAE** representa el error absoluto medio de las predicciones.
- **R²** indica qué proporción de la variabilidad del precio objetivo es explicada por el modelo.


## 8. Guardar el modelo

El modelo completo, incluyendo el preprocesamiento, se guarda para poder utilizarlo posteriormente sin tener que entrenarlo de nuevo.

In [78]:
ruta_modelo = "modelo_entrenado.pkl"
joblib.dump(modelo, ruta_modelo)

print(f"Modelo guardado en: {ruta_modelo}")


Modelo guardado en: modelo_entrenado.pkl


## 9. Comprobación final

Se realiza una predicción de ejemplo usando una fila del conjunto de prueba.

In [79]:
ejemplo = X_test.iloc[[0]]
prediccion_ejemplo = modelo.predict(ejemplo)[0]
valor_real = y_test.iloc[0]

print(f"Precio real: {valor_real:.4f}")
print(f"Precio predicho: {prediccion_ejemplo:.4f}")


Precio real: 12.5000
Precio predicho: 12.5000


## Conclusión

Se desarrolló un modelo de regresión basado en Random Forest para
predecir `unit_price`. Se eliminaron variables de identificación,
variables temporales y `total_price`, debido a su posible relación
directa con la variable objetivo. También se excluyó `pizza_name`,
ya que identifica de forma muy específica el producto y podía hacer
que el modelo memorizara su precio.

El modelo fue evaluado mediante RMSE, MAE y R², obteniendo:

- RMSE: 0.763985
- MAE: 0.135845
- R²: 0.955790

Finalmente, el modelo entrenado fue almacenado en
`modelo_entrenado.pkl`.